In [1]:
import glob
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
#import tifffile
import cv2
from PIL import Image

from skimage.color import label2rgb
import rasterio
from rasterio.enums import ColorInterp

import imageio.v3 as imageio

from tqdm import tqdm

# Set up calibration data spreadsheet

In [2]:
data = pd.read_csv('calibration_data/calibration_points.csv')

In [3]:
data = data[data["point_id"]<=1200]

In [4]:
data["image_filename"] = data["point_id"].apply(lambda i: f'calibration_data/images/{int(i):04d}.tif')
data["image_with_sampled_point_filename"] = data["point_id"].apply(lambda i: f'calibration_data/images_with_sampled_point/{int(i):04d}.tif')
data["image_with_sampled_point_visible_filename"] = data["point_id"].apply(lambda i: f'calibration_data/images_with_sampled_point_visible/{int(i):04d}.png')

In [5]:
image_with_sampled_point_filenames = data["image_with_sampled_point_filename"]
image_with_sampled_point_visible_filenames = data["image_with_sampled_point_visible_filename"]
sampled_coords = []

for i in tqdm(range(len(image_with_sampled_point_filenames))):
    image_with_sampled_point_filename = image_with_sampled_point_filenames[i]
    image_with_sampled_point_visible_filename = image_with_sampled_point_visible_filenames[i]
    image_with_sampled_point = imageio.imread(image_with_sampled_point_filename)[:, :, :3]
    sampled_coord = np.argwhere(image_with_sampled_point[:, :, 0]==300)[0]
    sampled_coords.append(sampled_coord)

data["sampled_coords"] = sampled_coords
data.to_csv('calibration_data/calibration_points_with_sampled_coords.csv', index=False)

100%|██████████| 1200/1200 [00:34<00:00, 34.79it/s]


In [6]:
data

,system:index,cultivated,point_id,.geo,image_filename,image_with_sampled_point_filename,image_with_sampled_point_visible_filename,sampled_coords
0,00000000000000000000,1,1.0,"{""type"":""Point"",""coordinates"":[-110.6651271743...",calibration_data/images/0001.tif,calibration_data/images_with_sampled_point/000...,calibration_data/images_with_sampled_point_vis...,"[566, 729]"
1,00000000000000000001,1,2.0,"{""type"":""Point"",""coordinates"":[-92.09303817323...",calibration_data/images/0002.tif,calibration_data/images_with_sampled_point/000...,calibration_data/images_with_sampled_point_vis...,"[689, 978]"
2,00000000000000000002,1,3.0,"{""type"":""Point"",""coordinates"":[-88.80564172629...",calibration_data/images/0003.tif,calibration_data/images_with_sampled_point/000...,calibration_data/images_with_sampled_point_vis...,"[700, 677]"
3,00000000000000000003,1,4.0,"{""type"":""Point"",""coordinates"":[-100.9435466670...",calibration_data/images/0004.tif,calibration_data/images_with_sampled_point/000...,calibration_data/images_with_sampled_point_vis...,"[931, 632]"
4,00000000000000000004,1,5.0,"{""type"":""Point"",""coordinates"":[-97.04459342882...",calibration_data/images/0005.tif,calibration_data/images_with_sampled_point/000...,calibration_data/images_with_sampled_point_vis...,"[1015, 852]"
...,...,...,...,...,...,...,...,...
1195,000000000000000004ab,1,1196.0,"{""type"":""Point"",""coordinates"":[-86.21786485734...",calibration_data/images/1196.tif,calibration_data/images_with_sampled_point/119...,calibration_data/images_with_sampled_point_vis...,"[821, 965]"
1196,000000000000000004ac,1,1197.0,"{""type"":""Point"",""coordinates"":[-108.5211245013...",calibration_data/images/1197.tif,calibration_data/images_with_sampled_point/119...,calibration_data/images_with_sampled_point_vis...,"[757, 740]"
1197,000000000000000004ad,1,1198.0,"{""type"":""Point"",""coordinates"":[-90.60041206390...",calibration_data/images/1198.tif,calibration_data/images_with_sampled_point/119...,calibration_data/images_with_sampled_point_vis...,"[534, 721]"
1198,000000000000000004ae,1,1199.0,"{""type"":""Point"",""coordinates"":[-102.1252307417...",calibration_data/images/1199.tif,calibration_data/images_with_sampled_point/119...,calibration_data/images_with_sampled_point_vis...,"[966, 635]"


# Create images with sampled point visible - these will be used for labeling

In [7]:
image_with_sampled_point_filenames = data["image_with_sampled_point_filename"]
image_with_sampled_point_visible_filenames = data["image_with_sampled_point_visible_filename"]
sampled_coords = []

for i in tqdm(range(len(image_with_sampled_point_filenames))):
    image_with_sampled_point_filename = image_with_sampled_point_filenames[i]
    image_with_sampled_point_visible_filename = image_with_sampled_point_visible_filenames[i]
    image_with_sampled_point = imageio.imread(image_with_sampled_point_filename)[:, :, :3]
    sampled_coord = np.argwhere(image_with_sampled_point[:, :, 0]==300)[0]
    sampled_coords.append(sampled_coord)
    
    x, y = sampled_coord
    d=3
    image_with_sampled_point[x-d:x+d+1, y-d:y+d+1] = [255, 0, 0]
    #image_with_sampled_point[x-d:x+d+1, y-d] = [255, 0, 0]
    #image_with_sampled_point[x-d:x+d+1, y+d] = [255, 0, 0]
    #image_with_sampled_point[x-d, y-d:y+d+1] = [255, 0, 0]
    #image_with_sampled_point[x+d, y-d:y+d+1] = [255, 0, 0]
    image_with_sampled_point[x, y] = [0, 0, 0]
    
    im = Image.fromarray(image_with_sampled_point.astype(np.uint8), 'RGB')
    im.save(image_with_sampled_point_visible_filename)

100%|██████████| 1200/1200 [14:00<00:00,  1.43it/s]
